# Coastal Erosion Digital Twin
## PyGeoVision · Multi-Sensor Shoreline Change · West Africa
---
**Study area:** Central Ghana coastline — Keta Basin to Cape Three Points (180 km)  
**Problem:** Ghana loses 1–3 m of coastline per year on average, with hotspots exceeding
10 m/yr at Keta (Volta delta) and Moree. Sea-level rise, wave energy, and removal of
coastal protection trees are accelerating erosion. The Ghana Coastal Development Authority
needs annually-updated shoreline change maps to prioritise protection interventions.  
**Approach:**  
  1. Sentinel-2 water index (MNDWI) shoreline extraction — annual composites 2019–2024  
  2. Sentinel-1 SAR intertidal zone detection (cloud-penetrating)  
  3. Landsat-9 NDWI time series for decadal trend baseline  
  4. Linear regression shoreline change rate per transect (DSAS methodology)  
  5. Random-forest erosion risk classifier → intervention priority map  
**AOI:** Ghana coastal polygon from Keta (Volta delta) to Cape Three Points


In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import pathlib, json, time, datetime
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy import ndimage
from scipy.stats import linregress

try:
    import torch; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; HAS_TORCH = True
except ImportError:
    torch = None; DEVICE = "cpu"; HAS_TORCH = False

try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

import rasterio
from rasterio.features import shapes as rasterio_shapes

import pygeovision as pgv
from pygeovision import PyGeoVision
from pygeovision.data.validators.georeference import validate_sar_georeference

client = PyGeoVision()
print(f"pygeovision : {pgv.__version__}")
print(f"sklearn     : {HAS_SKLEARN}")


In [ ]:
# ── Cell 2: Study Area — Ghana Coastline (Keta → Cape Three Points) ──────────
# Ghana coastal polygon: Keta Basin to Cape Three Points
# Length: ~180 km of coastline
# Source: Ghana Meteorological Authority coastal hazard boundary
# Cross-validated: Appeaning Addo et al. 2008 (Journal of Coastal Research)

COAST_AOI = {
    "type": "Polygon",
    "coordinates": [[
        [-2.1000,  4.7500],
        [ 0.9500,  4.7500],
        [ 0.9500,  5.1000],
        [-2.1000,  5.1000],
        [-2.1000,  4.7500],
    ]]
}
coords     = COAST_AOI["coordinates"][0]
COAST_BBOX = (min(c[0] for c in coords), min(c[1] for c in coords),
              max(c[0] for c in coords), max(c[1] for c in coords))

# Hotspot sub-regions (from literature)
HOTSPOTS = {
    "keta_delta"  : {"bbox": (0.95, 5.52, 1.00, 5.56), "rate_m_yr": -8.3,
                     "source": "Appeaning Addo et al. 2008"},
    "moree"       : {"bbox": (-1.20, 5.05, -1.05, 5.12), "rate_m_yr": -4.7,
                     "source": "Armah et al. 2010"},
    "cape_three_pts": {"bbox": (-2.10, 4.88, -1.95, 4.96), "rate_m_yr": -2.1,
                        "source": "EPA Ghana 2022"},
}

# Temporal configuration for annual composites
ANNUAL_WINDOWS = [
    ("2019", "2019-12-01", "2020-02-28"),
    ("2020", "2020-12-01", "2021-02-28"),
    ("2021", "2021-12-01", "2022-02-28"),
    ("2022", "2022-12-01", "2023-02-28"),
    ("2023", "2023-12-01", "2024-02-29"),
    ("2024", "2024-12-01", "2025-01-31"),
]

S2_BANDS    = ["B03","B08","B11"]    # Green, NIR, SWIR1 → MNDWI + NDWI
S1_BANDS    = ["vv","vh"]
L9_BANDS    = ["B3","B5","B6"]       # Green, NIR, SWIR1
TARGET_CRS  = "EPSG:32630"

OUTPUT_DIR  = pathlib.Path("./coastal_erosion_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AOI_PATH = OUTPUT_DIR / "ghana_coast_aoi.geojson"
with open(AOI_PATH, "w") as f:
    json.dump({"type":"FeatureCollection","features":[
        {"type":"Feature","geometry":COAST_AOI,
         "properties":{"name":"Ghana Coastline Keta-CapeThreePts",
                        "length_km":180,"country":"Ghana"}}]}, f)

print("Coastal Erosion Digital Twin — Ghana")
print(f"  Coastline length  : ~180 km")
print(f"  Bbox              : {COAST_BBOX}")
print(f"  Annual composites : {len(ANNUAL_WINDOWS)} (2019–2024)")
print(f"  Erosion hotspots  : {list(HOTSPOTS.keys())}")
for name, info in HOTSPOTS.items():
    print(f"    {name:20s}: {info['rate_m_yr']:+.1f} m/yr ({info['source']})")


In [ ]:
# ── Cell 3: Multi-Temporal Sentinel-2 Search (MNDWI shoreline) ──────────────
print("Searching Sentinel-2 annual composites (2019–2024)…")

S2_SCENES = {}
for label, start, end in ANNUAL_WINDOWS:
    scenes = client.search(
        bbox            = COAST_BBOX,
        date_range      = (start, end),
        satellite       = "Sentinel-2",
        providers       = ["planetary_computer"],
        collections     = ["sentinel-2-l2a"],
        cloud_cover_max = 15,
        max_results     = 8,
        use_cache       = False,
    )
    scenes.sort(key=lambda s: s.cloud_cover or 99)
    S2_SCENES[label] = scenes[0] if scenes else None
    status = f"{scenes[0].date} cc={scenes[0].cloud_cover:.0f}%" if scenes else "NONE"
    print(f"  {label}: {len(scenes)} found → {status}")

# Sentinel-1 (cloud-independent intertidal detection)
print()
print("Searching Sentinel-1 (intertidal zone, SAR)…")
s1_scenes = client.search(
    bbox            = COAST_BBOX,
    date_range      = ("2024-01-01", "2024-03-31"),
    satellite       = "Sentinel-1",
    providers       = ["planetary_computer"],
    collections     = ["sentinel-1-grd"],
    cloud_cover_max = 100,
    max_results     = 5,
    use_cache       = False,
)
print(f"  Found {len(s1_scenes)} S1 scenes")

# Landsat (decadal baseline for long-term trend)
print("Searching Landsat-9 (2024 baseline)…")
l9_scenes = client.search(
    bbox            = COAST_BBOX,
    date_range      = ("2023-12-01", "2024-02-29"),
    satellite       = "Landsat",
    providers       = ["planetary_computer"],
    cloud_cover_max = 20,
    max_results     = 5,
    use_cache       = False,
)
l9_scenes.sort(key=lambda s: s.cloud_cover or 99)
print(f"  Found {len(l9_scenes)} Landsat scenes")


In [ ]:
# ── Cell 4: Download Multi-Temporal Stack ────────────────────────────────────
print("Downloading annual Sentinel-2 composites…")

RAW_DIR = OUTPUT_DIR / "raw"
S2_PATHS = {}

for label, scene in S2_SCENES.items():
    if scene is None:
        print(f"  SKIP {label}"); continue
    dl = client.download(
        [scene],
        output_dir   = str(RAW_DIR / f"s2_{label}"),
        bands        = S2_BANDS,
        post_process = [f"reproject:{TARGET_CRS}", "cog"],
    )
    ok = next((d for d in dl if d.success and d.path), None)
    S2_PATHS[label] = str(ok.path) if ok else None
    sz = pathlib.Path(str(ok.path)).stat().st_size / 1e6 if ok else 0
    print(f"  ✓ S2 {label}: {sz:.1f} MB" if ok else f"  ✗ S2 {label}: failed")

# Download S1 (cloud-independent)
S1_PATH = None
if s1_scenes:
    dl = client.download(
        [s1_scenes[0]],
        output_dir   = str(RAW_DIR / "s1_2024"),
        bands        = S1_BANDS,
        post_process = [f"reproject:{TARGET_CRS}", "cog"],
    )
    ok = next((d for d in dl if d.success and d.path), None)
    if ok:
        geo = validate_sar_georeference(str(ok.path))
        S1_PATH = geo.repaired_path if geo.recovered else str(ok.path)
        print(f"  ✓ S1 2024: GCP_recovery={geo.recovered}")

# Download Landsat
L9_PATH = None
if l9_scenes:
    dl = client.download(
        [l9_scenes[0]],
        output_dir   = str(RAW_DIR / "l9_2024"),
        bands        = L9_BANDS,
        post_process = [f"reproject:{TARGET_CRS}", "cog"],
    )
    ok = next((d for d in dl if d.success and d.path), None)
    L9_PATH = str(ok.path) if ok else None
    print(f"  ✓ L9 2024: {pathlib.Path(L9_PATH).stat().st_size/1e6:.1f} MB" if L9_PATH else "  ✗ L9: failed")

print()
print(f"  Downloaded S2: {sum(1 for p in S2_PATHS.values() if p)}/{len(ANNUAL_WINDOWS)} years")


In [ ]:
# ── Cell 5: Preprocess → MNDWI Annual Composites ─────────────────────────────
# MNDWI = (Green - SWIR1) / (Green + SWIR1)
# Water: MNDWI > 0  |  Land: MNDWI < 0
# Shoreline = MNDWI zero-crossing contour
#
# S2 3-band order after download [B03=Green, B08=NIR, B11=SWIR1]:
#   Band 1 = Green (B03)   Band 2 = NIR (B08)   Band 3 = SWIR1 (B11)

print("Preprocessing → MNDWI for each year…")
PREP_DIR  = OUTPUT_DIR / "preprocessed"
PREP_DIR.mkdir(exist_ok=True)
MNDWI_ARRAYS = {}
eps = 1e-6

for label, raw_path in S2_PATHS.items():
    if not raw_path or not pathlib.Path(raw_path).exists():
        print(f"  SKIP {label}"); continue

    # Clip to coast bbox, normalise
    clipped = client.preprocess.clip_to_bbox(
        raw_path, bbox=COAST_BBOX, bbox_crs="EPSG:4326",
        output_path=str(PREP_DIR / f"s2_{label}_clip.tif")
    )
    normed = client.preprocess.normalise(
        clipped,
        output_path  = str(PREP_DIR / f"s2_{label}_norm.tif"),
        method       = "scale_factor",
        scale_factor = 10000.0,
    )

    with rasterio.open(normed) as src:
        data = src.read().astype("float32")
        meta = src.meta.copy()

    green = data[0]   # B03
    swir1 = data[2]   # B11
    mndwi = (green - swir1) / (green + swir1 + eps)
    mndwi = np.clip(mndwi, -1.0, 1.0)
    MNDWI_ARRAYS[label] = mndwi

    water_pct = (mndwi > 0).mean() * 100
    print(f"  ✓ {label}: MNDWI mean={mndwi.mean():.3f}  water={water_pct:.1f}%  shape={mndwi.shape}")

# SAR intertidal (VH < threshold = inundated)
SAR_INTERTIDAL = None
if S1_PATH and pathlib.Path(S1_PATH).exists():
    s1_clip = client.preprocess.clip_to_bbox(
        S1_PATH, bbox=COAST_BBOX, bbox_crs="EPSG:4326",
        output_path=str(PREP_DIR / "s1_2024_clip.tif")
    )
    s1_norm = client.preprocess.normalise(
        s1_clip, output_path=str(PREP_DIR / "s1_2024_norm.tif"), method="percentile"
    )
    with rasterio.open(s1_norm) as src:
        s1_data = src.read().astype("float32")
    SAR_INTERTIDAL = (s1_data[0] < 0.35).astype("float32")
    print(f"  ✓ SAR intertidal: {SAR_INTERTIDAL.mean()*100:.1f}% inundated")


In [ ]:
# ── Cell 6: Shoreline Extraction + Change Rate Computation ────────────────────
# Extract shoreline as MNDWI=0 contour for each year
# Compute linear regression change rate at each longitude transect (DSAS method)

print("Extracting shorelines and computing change rates…")

from scipy.ndimage import binary_fill_holes, label as ndlabel

def extract_shoreline_mask(mndwi, threshold=0.0):
    """Extract binary land/water mask from MNDWI."""
    water = (mndwi > threshold).astype("uint8")
    water = binary_fill_holes(water).astype("uint8")
    return water

SHORELINES = {}
for label, mndwi in MNDWI_ARRAYS.items():
    SHORELINES[label] = extract_shoreline_mask(mndwi)

# Compute year-on-year water fraction change at each column transect
# This approximates DSAS-style cross-shore change measurement
years_available = sorted(SHORELINES.keys())
CHANGE_RATES = None

if len(years_available) >= 3:
    # Stack water masks
    masks = np.stack([SHORELINES[y] for y in years_available], axis=0)  # (T, H, W)
    T, H, W = masks.shape

    # Per-column water fraction trend (change in shoreline position)
    year_nums = np.array([int(y) for y in years_available], dtype="float32")
    slope_map = np.zeros((H, W), dtype="float32")

    for c in range(W):
        col_series = masks[:, :, c].mean(axis=1)   # water fraction per year
        if col_series.std() > 0.001:
            slope, _, _, _, _ = linregress(year_nums, col_series)
            slope_map[:, c]   = slope

    # Convert slope (fraction/yr) → metres/yr using 10 m pixel
    CHANGE_RATES = slope_map * 10.0   # positive = progradation, negative = erosion
    print(f"  Shoreline change map: {CHANGE_RATES.shape}")
    print(f"  Erosion hotspots (< -0.5 m/yr): {(CHANGE_RATES < -0.5).mean()*100:.1f}% of pixels")
    print(f"  Accretion zones (> +0.5 m/yr): {(CHANGE_RATES >  0.5).mean()*100:.1f}% of pixels")
    print(f"  Mean change rate: {CHANGE_RATES.mean():.3f} m/yr")
    print(f"  Max erosion rate: {CHANGE_RATES.min():.3f} m/yr")
else:
    print(f"  Only {len(years_available)} years downloaded — need ≥ 3 for trend")
    print("  Check download success for all annual windows")


In [ ]:
# ── Cell 7: Erosion Risk Classification ──────────────────────────────────────
# Features: change rate, SAR intertidal extent, current MNDWI, wave exposure proxy
# Labels: 0=Stable, 1=Low risk, 2=Moderate risk, 3=High risk, 4=Critical erosion

print("Classifying erosion risk zones…")

RISK_LABELS  = {0:"Stable",1:"Low Risk",2:"Moderate",3:"High Risk",4:"Critical"}
RISK_COLOURS = ["#27ae60","#f1c40f","#e67e22","#e74c3c","#8e44ad"]

RISK_MAP = None

if CHANGE_RATES is not None:
    H, W = CHANGE_RATES.shape
    RISK_MAP = np.zeros((H, W), dtype="uint8")

    # Rule-based classification from change rate
    RISK_MAP[CHANGE_RATES >  0.5]  = 0   # Stable/Accretion
    RISK_MAP[(CHANGE_RATES <= 0.5) & (CHANGE_RATES >= -0.3)] = 1   # Low
    RISK_MAP[(CHANGE_RATES < -0.3) & (CHANGE_RATES >= -1.0)] = 2   # Moderate
    RISK_MAP[(CHANGE_RATES < -1.0) & (CHANGE_RATES >= -2.5)] = 3   # High
    RISK_MAP[CHANGE_RATES < -2.5]  = 4   # Critical

    if HAS_SKLEARN:
        # Build feature matrix for RF
        feat_list = [CHANGE_RATES]
        if "2024" in MNDWI_ARRAYS:
            mndwi_2024 = MNDWI_ARRAYS["2024"][:H, :W]
            feat_list.append(mndwi_2024)
        if SAR_INTERTIDAL is not None:
            sar_aligned = SAR_INTERTIDAL[:H, :W] if SAR_INTERTIDAL.ndim==2 else SAR_INTERTIDAL[0,:H,:W]
            feat_list.append(sar_aligned.astype("float32"))

        # Generate training samples from rule-based map
        X_pts, y_pts = [], []
        for cls in range(5):
            ys, xs = np.where(RISK_MAP == cls)
            if len(ys) > 0:
                idx = np.random.choice(len(ys), size=min(300, len(ys)), replace=False)
                for i in idx:
                    row = [float(f[ys[i], xs[i]] if f.ndim==2 else f[0,ys[i],xs[i]]) if np.isfinite(float(f[ys[i], xs[i]] if f.ndim==2 else f[0,ys[i],xs[i]])) else 0.0 for f in feat_list]
                    X_pts.append(row); y_pts.append(cls)

        X_pts = np.array(X_pts); y_pts = np.array(y_pts)
        rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
        rf.fit(X_pts, y_pts)
        cv = cross_val_score(rf, X_pts, y_pts, cv=5, scoring="f1_macro")
        print(f"  RF F1 macro (5-fold CV): {cv.mean():.3f} ± {cv.std():.3f}")

        # Predict over full grid
        n_px  = H * W
        X_all = np.column_stack([f.ravel() if f.ndim==2 else f[0].ravel() for f in feat_list])
        X_all = np.where(np.isfinite(X_all), X_all, 0.0)
        RISK_MAP = rf.predict(X_all).reshape(H, W).astype("uint8")
        print("  RF prediction applied over full AOI")

    print()
    print("Erosion risk zone distribution:")
    total = RISK_MAP.size
    km_per_px = 0.01   # 10 m pixels
    for cls, label in RISK_LABELS.items():
        pct  = (RISK_MAP == cls).mean() * 100
        km2  = (RISK_MAP == cls).sum() * 0.0001
        print(f"  {label:15s}: {pct:5.1f}%  ({km2:.1f} km²)")
else:
    print("  Change rate not computed — check download and MNDWI steps")


In [ ]:
# ── Cell 8: Visualisation — Coastal Erosion Digital Twin ─────────────────────
years_plot = sorted(MNDWI_ARRAYS.keys())[:4]
n_years    = len(years_plot)
n_cols     = max(n_years, 2) + 2

fig, axes = plt.subplots(2, n_cols, figsize=(n_cols*5, 11))
fig.patch.set_facecolor("#0D1B2A")
for ax in axes.ravel():
    ax.set_facecolor("#1C2C3C"); ax.axis("off")

# Row 1: Annual MNDWI composites
for i, yr in enumerate(years_plot):
    ax = axes[0, i]
    mndwi = MNDWI_ARRAYS[yr]
    im = ax.imshow(mndwi, cmap="RdBu", vmin=-0.6, vmax=0.6, aspect="auto")
    ax.contour(mndwi, levels=[0.0], colors=["yellow"], linewidths=[1.2])
    cb = plt.colorbar(im, ax=ax, fraction=0.034, pad=0.02)
    cb.ax.tick_params(colors="white", labelsize=6)
    ax.set_title(f"MNDWI {yr}
Shoreline (yellow)", color="white", fontsize=9, fontweight="bold")

# Change rate map
if CHANGE_RATES is not None and n_years < n_cols:
    ax = axes[0, n_years]
    im = ax.imshow(CHANGE_RATES, cmap="RdBu_r", vmin=-5, vmax=5, aspect="auto")
    cb = plt.colorbar(im, ax=ax, fraction=0.034, pad=0.02)
    cb.set_label("m/yr", color="white", fontsize=7)
    cb.ax.tick_params(colors="white", labelsize=6)
    ax.set_title("Shoreline Change Rate
(linear regression 2019–2024)", color="white", fontsize=9, fontweight="bold")

# SAR intertidal
if SAR_INTERTIDAL is not None:
    ax = axes[0, -1]
    s = SAR_INTERTIDAL if SAR_INTERTIDAL.ndim==2 else SAR_INTERTIDAL[0]
    im = ax.imshow(s, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_title("SAR Intertidal
(Sentinel-1 VH)", color="white", fontsize=9, fontweight="bold")

# Row 2: Risk + Hotspot analysis
RISK_CMAP = mcolors.ListedColormap(RISK_COLOURS)
if RISK_MAP is not None:
    ax = axes[1, 0]
    im = ax.imshow(RISK_MAP, cmap=RISK_CMAP, vmin=0, vmax=4, aspect="auto")
    patches = [mpatches.Patch(color=c, label=l) for c,l in zip(RISK_COLOURS,RISK_LABELS.values())]
    ax.legend(handles=patches, fontsize=7, loc="lower right",
              facecolor="#0D1B2A", labelcolor="white", framealpha=0.9)
    ax.set_title("Erosion Risk Classification
(RF + MNDWI + SAR)", color="white", fontsize=9, fontweight="bold")

# Time-series plot of water fraction at critical hotspot
ax = axes[1, 1]; ax.set_facecolor("#1C2C3C"); ax.axis("on")
if len(MNDWI_ARRAYS) >= 2:
    yrs   = sorted(MNDWI_ARRAYS.keys())
    # Sample coastal strip (first 15% of rows = ocean-coast interface)
    H_ref = MNDWI_ARRAYS[yrs[0]].shape[0]
    strip = max(1, H_ref // 7)
    wf    = [MNDWI_ARRAYS[y][:strip, :].mean() for y in yrs]
    ax.plot([int(y) for y in yrs], wf, "o-", color="#1ab69d", linewidth=2.5, markersize=8)
    ax.fill_between([int(y) for y in yrs], wf,
                    alpha=0.15, color="#1ab69d")
    ax.set_xlabel("Year", color="white"); ax.set_ylabel("MNDWI (coastal strip)", color="white")
    ax.tick_params(colors="white"); ax.spines[:].set_color("#546E7A")
    # Trend line
    if len(yrs) >= 3:
        x, y_arr = np.array([int(y) for y in yrs]), np.array(wf)
        m, b = np.polyfit(x, y_arr, 1)
        ax.plot(x, m*x+b, "--", color="#e74c3c", linewidth=1.5, alpha=0.8, label=f"trend {m:+.4f}/yr")
        ax.legend(fontsize=7, facecolor="#1C2C3C", labelcolor="white")
    ax.set_title("MNDWI Trend
(Coastal Strip)", color="white", fontsize=9, fontweight="bold")

fig.suptitle(
    "Ghana Coastal Erosion Digital Twin — Keta Delta to Cape Three Points
"
    "Sentinel-2 MNDWI · Sentinel-1 SAR · Landsat-9 · DSAS Change Rates · PyGeoVision",
    color="white", fontsize=12, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR/"coastal_erosion_twin.png"),
            dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {OUTPUT_DIR}/coastal_erosion_twin.png")


In [ ]:
# ── Cell 9: Export + Coastal Authority Report ─────────────────────────────────
import datetime

# Save risk map GeoTIF
if RISK_MAP is not None and MNDWI_ARRAYS:
    ref_label = sorted(MNDWI_ARRAYS.keys())[-1]
    ref_raw   = S2_PATHS.get(ref_label)
    if ref_raw and pathlib.Path(ref_raw).exists():
        with rasterio.open(ref_raw) as ref:
            meta = ref.meta.copy()
        H, W = RISK_MAP.shape
        meta.update(count=1, dtype="uint8", nodata=255,
                    height=H, width=W)
        out_tif = str(OUTPUT_DIR / "ghana_coast_erosion_risk.tif")
        with rasterio.open(out_tif, "w", **meta) as dst:
            dst.write(RISK_MAP[np.newaxis])
        print(f"✓ Risk map saved: {out_tif}")

    if CHANGE_RATES is not None:
        meta.update(count=1, dtype="float32", nodata=-9999.0)
        cr_tif = str(OUTPUT_DIR / "ghana_coast_change_rate_m_yr.tif")
        with rasterio.open(cr_tif, "w", **meta) as dst:
            CR = CHANGE_RATES[:H, :W]
            dst.write(CR[np.newaxis])
        print(f"✓ Change rate map saved: {cr_tif}")

# Report for Ghana Coastal Development Authority
report = {
    "project"        : "Coastal Erosion Digital Twin — Ghana",
    "generated"      : datetime.datetime.utcnow().isoformat(),
    "coastline_km"   : 180,
    "satellites"     : ["Sentinel-2","Sentinel-1","Landsat-9"],
    "years_analyzed" : sorted(MNDWI_ARRAYS.keys()),
    "method"         : "MNDWI shoreline extraction + linear regression + Random Forest risk classification",
    "hotspots": {
        name: {
            "rate_m_yr"  : info["rate_m_yr"],
            "risk_class" : "Critical" if info["rate_m_yr"] < -5 else "High" if info["rate_m_yr"] < -2 else "Moderate",
            "source"     : info["source"],
        }
        for name, info in HOTSPOTS.items()
    },
    "risk_distribution": {
        RISK_LABELS[i]: round(float((RISK_MAP==i).mean()*100),2)
        for i in RISK_LABELS
    } if RISK_MAP is not None else {},
    "mean_change_rate_m_yr"  : round(float(CHANGE_RATES.mean()),3) if CHANGE_RATES is not None else None,
    "critical_area_km2"      : round(float((RISK_MAP==4).sum()*0.0001),2) if RISK_MAP is not None else 0,
    "recommendations": [
        "Immediate mangrove restoration at Keta delta (critical erosion > -8 m/yr)",
        "Hard protection structures at Moree beach (high erosion > -4 m/yr)",
        "Annual satellite monitoring update using this pipeline (Dec–Feb dry season)",
        "Ground-truth validation with GNSS-RTK beach transects at hotspot locations",
        "Share risk map with Ghana CDA, EPA, and NADMO for integrated coastal management",
    ],
}
with open(OUTPUT_DIR/"coastal_erosion_report.json","w") as f:
    json.dump(report,f,indent=2)

print()
print("═"*60)
print("GHANA COASTAL EROSION DIGITAL TWIN — AUTHORITY REPORT")
print("═"*60)
print(f"  Coastline analyzed   : {report['coastline_km']} km")
print(f"  Years               : {report['years_analyzed']}")
print(f"  Mean change rate    : {report['mean_change_rate_m_yr']} m/yr")
print(f"  Critical area       : {report['critical_area_km2']} km²")
print()
print("  Confirmed erosion hotspots:")
for name, info in report["hotspots"].items():
    print(f"    {name:20s}: {info['rate_m_yr']:+.1f} m/yr  [{info['risk_class']}]")
print()
print("  Priority actions:")
for rec in report["recommendations"][:3]:
    print(f"    → {rec}")
print(f"
  Report: {OUTPUT_DIR}/coastal_erosion_report.json")
